# 03 - The MusicBrainz genre taxonomy

**Goal of this notebook:** find out which genres are subgenres of which, so we can group our genre labels into families (rock, electronic, jazz, ...).

MusicBrainz records relationships between genres. We found the table in the MusicBrainz dump (`l_genre_genre`) and extracted it into the folder `MusicBrainz genre taxonomy`. Steps:

1. **Read the tables** that describe the relationships (all small).
2. **Join them**: relationship -> its type (subgenre / fusion of / influenced by) -> the genre names.
3. **Build the hierarchy**: for every genre its parent, its family (the top genre above it) and its depth.
4. **Match it to our genre counts** (from notebook 02) and see how much of our listening is covered.

Nothing in the MusicBrainz tables or in our earlier results is changed. This notebook only creates two new csv files in the `MusicBrainz genre taxonomy` folder.

## 1. Setup

In [ ]:
%pip install -q duckdb

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

RAW = Path("/Users/hanshornbek/Desktop/Projects/Data Mining project")
MB = RAW / "Raw data/MusicBrainz Full/tables/mbdump"                  # the MusicBrainz database tables
TAX = Path.cwd().parent / "MusicBrainz genre taxonomy"                # the extracted table l_genre_genre is here; our results go here too
COUNTS = Path.cwd().parent / "Output_dev" / "genre_counts"                # the genre counts per year from notebook 02
SUFFIX = "dev"                                                          # must match the SUFFIX used in notebook 02

con = duckdb.connect()

def run(sql):
    """Run a SQL statement that creates something and finish it completely."""
    con.execute(sql).fetchall()

def tsv(table):
    """Read a MusicBrainz table. These files are tab-separated, have no header row, and write a missing value
    as backslash-N. The columns get the names c0, c1, c2, ... (in the order of the MusicBrainz documentation)."""
    path = table if isinstance(table, Path) else MB / table
    n_columns = len(open(path, encoding="utf-8", errors="ignore").readline().rstrip("\n").split("\t"))
    names = ", ".join(f"'c{i}'" for i in range(n_columns))
    return f"read_csv('{path}', header=false, delim='\\t', quote='', escape='', nullstr='\\\\N', all_varchar=true, names=[{names}])"

print("l_genre_genre found:", (TAX / "l_genre_genre").exists())

## 2. Read the tables

- `genre`: every official genre with a number (`id`) and its name. These are the same 2,221 names we use for our counts.
- `link_type`: the kinds of relationships. We only want the ones between two genres.
- `link`: one row per relationship, with its type. This table is big, so we keep only the rows of the genre types.
- `l_genre_genre`: the relationships themselves: `entity0` and `entity1` are two genre ids, and `link` points into the `link` table.

In [ ]:
run(f"CREATE TABLE genre AS SELECT c0::BIGINT AS id, c2 AS name FROM {tsv('genre')}")
print(f"{con.execute('SELECT count(*) FROM genre').fetchall()[0][0]:,} genres")

# column 4 and 5 of link_type say which kinds of things the relationship connects, column 6 is the name
run(f"CREATE TABLE link_type AS SELECT c0::BIGINT AS id, c6 AS name FROM {tsv('link_type')} WHERE c4 = 'genre' AND c5 = 'genre'")
con.sql("SELECT * FROM link_type ORDER BY id").df()

In [ ]:
run(f"""CREATE TABLE link AS
        SELECT c0::BIGINT AS id, c1::BIGINT AS link_type FROM {tsv('link')}
        WHERE c1::BIGINT IN (SELECT id FROM link_type)""")
run(f"CREATE TABLE genre_genre AS SELECT c1::BIGINT AS link, c2::BIGINT AS entity0, c3::BIGINT AS entity1 FROM {tsv(TAX / 'l_genre_genre')}")

# how many relationships of each type?
con.sql("""
    SELECT lt.name AS relationship, count(*) AS pairs
    FROM genre_genre gg
    JOIN link l ON l.id = gg.link
    JOIN link_type lt ON lt.id = l.link_type
    GROUP BY 1 ORDER BY 2 DESC
""").df()

## 3. Which side is the parent?

For a "subgenre" relationship it is not obvious whether `entity0` or `entity1` is the parent. We look at pairs we know: `jazz` should be the parent of `crossover jazz`.

In [ ]:
con.sql("""
    SELECT lt.name AS relationship, g0.name AS entity0, g1.name AS entity1
    FROM genre_genre gg
    JOIN link l ON l.id = gg.link
    JOIN link_type lt ON lt.id = l.link_type
    JOIN genre g0 ON g0.id = gg.entity0
    JOIN genre g1 ON g1.id = gg.entity1
    WHERE g1.name IN ('crossover jazz', 'post-classical', 'melodic hardcore')
    ORDER BY 1, 3
""").df()

So for a subgenre relationship `entity0` is the **parent** and `entity1` the **subgenre**. We only use the subgenre relationships for the hierarchy (the "influenced by" and "fusion of" relationships are not parent/child).

## 4. The hierarchy

In [ ]:
# every parent -> child pair, with names
run("""
    CREATE TABLE sub AS
    SELECT g0.name AS parent, g1.name AS child
    FROM genre_genre gg
    JOIN link l ON l.id = gg.link AND l.link_type = (SELECT id FROM link_type WHERE name = 'subgenre')
    JOIN genre g0 ON g0.id = gg.entity0
    JOIN genre g1 ON g1.id = gg.entity1
""")
print(f"{con.execute('SELECT count(*) FROM sub').fetchall()[0][0]:,} parent-child pairs")

# a few genres have more than one parent. To keep one clear chain we use ONE parent per genre:
# the first one in alphabetical order. (all_parents keeps the full list, so nothing is hidden.)
run("""
    CREATE TABLE prim AS
    SELECT child AS genre, min(parent) AS parent, count(DISTINCT parent) AS n_parents,
           string_agg(DISTINCT parent, ' | ') AS all_parents
    FROM sub GROUP BY child
""")
con.sql("SELECT genre, all_parents FROM prim WHERE n_parents > 1 ORDER BY genre").df()

Now follow the parents upward until a genre has no parent. The last one is the **family** of the genre, and the number of steps is its **depth** (a genre directly under a top genre has depth 1). This is a *recursive* query: it starts with each genre's parent and keeps adding the parent of the parent.

In [ ]:
run("""
    CREATE TABLE tops AS
    WITH RECURSIVE up(genre, ancestor, depth, path) AS (
        SELECT genre, parent, 1, [genre, parent] FROM prim
        UNION ALL
        SELECT up.genre, p.parent, up.depth + 1, list_append(up.path, p.parent)
        FROM up JOIN prim p ON p.genre = up.ancestor
        WHERE NOT list_contains(up.path, p.parent)          -- never walk in a circle
    )
    SELECT genre, arg_max(ancestor, depth) AS family, max(depth) AS depth FROM up GROUP BY genre
""")

# one row for every one of the 2,221 genres. A genre without a parent is its own family (depth 0).
hierarchy = con.sql("""
    SELECT g.name                          AS genre,
           p.parent                        AS parent,
           p.all_parents                   AS all_parents,
           coalesce(p.n_parents, 0)        AS n_parents,
           coalesce(t.family, g.name)      AS family,
           coalesce(t.depth, 0)            AS depth,
           coalesce(c.n_children, 0)       AS n_children
    FROM genre g
    LEFT JOIN prim p ON p.genre = g.name
    LEFT JOIN tops t ON t.genre = g.name
    LEFT JOIN (SELECT parent AS genre, count(DISTINCT child) AS n_children FROM sub GROUP BY 1) c ON c.genre = g.name
    ORDER BY family, depth, genre
""").df()

hierarchy.to_csv(TAX / "genre_hierarchy.csv", index=False)
print(f"saved genre_hierarchy.csv ({len(hierarchy):,} genres)")
hierarchy.head(12)

In [ ]:
# how deep is it, and which families are the biggest?
print(hierarchy["depth"].value_counts().sort_index().rename("genres").to_frame().T.to_string())
print()
families = hierarchy[hierarchy["depth"] > 0].groupby("family").size().sort_values(ascending=False).rename("subgenres")
families.head(12).to_frame()

## 5. How much of our listening does the hierarchy cover?

Now we stack the genre counts of all years from notebook 02 and look up each of our genres in the hierarchy. Every genre of ours falls in one of three groups:

1. **has a parent** - it is a subgenre of something (for example `alternative rock`),
2. **top-level genre with subgenres** - it has no parent but has subgenres (for example `rock`),
3. **no relationships** - MusicBrainz knows no parent and no subgenres for it.

Groups 1 and 2 can be placed in a family. Group 3 is its own family.

In [ ]:
files = sorted(COUNTS.glob(f"*/genre_counts_*_{SUFFIX}.csv"))
ours = pd.concat(pd.read_csv(f) for f in files)
print(f"{len(files)} yearly files, {ours['year'].min()} to {ours['year'].max()}, {ours['genre'].nunique():,} different genres")

merged = ours.merge(hierarchy[["genre", "family", "depth", "n_children"]], on="genre", how="left")
print("genres of ours that are not in the MusicBrainz genre table:", merged["family"].isna().sum())

merged["category"] = "3 no relationships"
merged.loc[(merged["depth"] == 0) & (merged["n_children"] > 0), "category"] = "2 top-level genre with subgenres"
merged.loc[merged["depth"] > 0, "category"] = "1 has a parent (is a subgenre)"

coverage = (merged.groupby(["year", "category"])
                  .agg(genres=("genre", "nunique"), listens=("listens", "sum"))
                  .reset_index())
coverage.to_csv(TAX / "taxonomy_coverage.csv", index=False)

# all years together
total = merged.groupby("category").agg(listens=("listens", "sum"))
labels = merged.groupby("category")["genre"].nunique().rename("different_genres")
total = total.join(labels)
total["share_of_listens_%"] = (total["listens"] / total["listens"].sum() * 100).round(1)
total

In [ ]:
# which families have the most listening in our data (all years)?
by_family = merged.groupby("family")["listens"].sum().sort_values(ascending=False)
(by_family / by_family.sum() * 100).round(1).rename("share of listens (%)").head(15).to_frame()

## What to notice

- The hierarchy is MusicBrainz's own, so the genre names match ours exactly (no fuzzy matching).
- A few genres have more than one parent. We use the first parent in alphabetical order for the family and keep the full list in `all_parents`.
- `taxonomy_coverage.csv` has the coverage per year, so we can see whether it changes over time.
- **Next:** the family of each genre can be joined onto the genre counts (or later the per-user table) to compare families and to look at how subgenres move inside a family over the years.